# Introducción a la Ciencia de Datos: Tarea 2

Este notebook contiene el código de base para realizar la Tarea 2 del curso. Es la continuación de la Tarea 1, por lo que se utilizarán los mismos datos y se puede reutilizar cualquier parte del código de dicha tarea.

Puede copiar este notebook en su propio repositorio y trabajar sobre el mismo.
Las **instrucciones para ejecutar el notebook** están en la [página inicial del repositorio](https://gitlab.fing.edu.uy/maestria-cdaa/intro-cd).

Se utiliza el lenguaje Python y las librerías Pandas y scikit-learn. Para esta tarea se recomienda consultar la sección [Extracting features from text files](https://scikit-learn.org/stable/tutorial/text_analytics/working_with_text_data.html) de la documentación oficial de scikit-learn.

Recuerde que **se espera que no sea necesario revisar el código para corregir la tarea**, ya que todos los resultados y análisis relevantes deberían estar en el **informe en formato PDF**.

## Cargar bibliotecas (dependencias)
Recuerde instalar los requerimientos (`requirements.txt`) en el mismo entorno donde está ejecutando este notebook.

In [ ]:
from time import time
from pathlib import Path

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)

## Descarga del dataset
Se utilizan los mismos datos que en la Tarea 1. Ejecute la siguiente celda para descargar los datos y cargarlos en un dataframe de pandas.

In [ ]:
ds = load_dataset("tomas-gr/all-the-news-2-1-Component-one-sampled", split="train", cache_dir="../data")
df = ds.to_pandas()

# Tres medios con mayor cantidad de artículos
# Cantidad de articulos por medio de prensa:
print("Cantidad de artículos por medio de prensa:")
print(df['publication'].value_counts())

# Tome los 3 medios con más artículos
top3 = df['publication'].dropna().value_counts().head(3).index
print('Los 3 medios con más artículos son:')
print(top3)


# Parte 1: Dataset y representación numérica de texto

## 1. Preparación del dataset
Se utilizará un conjunto de datos reducido de los **tres medios de prensa con mayor cantidad de artículos**.
Se espera que utilice su propia versión de la función `clean_text()` de la Tarea 1.

Particione los datos para generar un conjunto de test del 30% del total, utilizando muestreo estratificado.

**Sugerencia**: utilice el parámetro `stratify` de la función `train_test_split` de scikit-learn y fije también el valor de `random_state` para obtener resultados reproducibles.

In [ ]:
# Cantidad de articulos de The New York Times que tienen la plantilla '(Reuters) -' en el texto del artículo
cantidad_articulos = df[(df['publication'] == 'The New York Times') & (df['article'].str.contains(r'\(Reuters\) -'))].shape[0]
print(f"\nCantidad de artículos de The New York Times que contienen la plantilla '(Reuters) -': {cantidad_articulos}")

# Cantidad de articulos de CNBC que tienen la plantilla '(Reuters) -' en el texto del artículo
cantidad_articulos_cnbc = df[(df['publication'] == 'CNBC') & (df['article'].str.contains(r'\(Reuters\)'))].shape[0]
print(f"\nCantidad de artículos de CNBC que contienen la plantilla '(Reuters) -': {cantidad_articulos_cnbc}")


In [ ]:
def remover_plantillas(df):

    df = df.copy()

    mask = df['publication'].isin(['Reuters', 'CNBC'])

    df.loc[mask, 'article'] = df.loc[mask, 'article'].str.replace(
        r'^(.*?)(\((Reuters|CNBC)\)\s*-)',
        '',
        regex=True
    )

    return df

In [ ]:
import re

def clean_text(series):

    text = series.fillna("").str.lower()

    # expandir contracciones
    contractions = {
        "i'm": "i am",
        "you're": "you are",
        "he's": "he is",
        "she's": "she is",
        "it's": "it is",
        "we're": "we are",
        "they're": "they are",
        "can't": "cannot",
        "won't": "will not",
        "don't": "do not",
        "doesn't": "does not"
    }

    for k, v in contractions.items():
        text = text.str.replace(k, v, regex=False)

    # reemplazar puntuación por espacios
    for punc in ["[", "\n", ",", ":", "?", "!", "(", ")", '"', "]", ";", ". "]:
        text = text.str.replace(punc, " ", regex=False)

    text = text.apply(lambda x: " ".join(
        w for w in x.split()
        if w.isalpha()   # SOLO letras puras
    ))

    # eliminar espacios múltiples
    text = text.str.replace(r"\s+", " ", regex=True).str.strip()

    return text

In [ ]:
# TODO: Obtenga los tres medios con mayor cantidad de artículos y filtre el DataFrame
top_3_publications = df['publication'].dropna().value_counts().head(3).index
df_top_3 = df[df['publication'].isin(top_3_publications)]
print('Los 3 medios con más artículos son:')
print(top_3_publications)

# Imprimir un ejemplo de articulo no antes de remover plantillas y luego de remover plantillas de Reuters
print("Ejemplo de artículo antes de remover plantillas:")
print(df_top_3.iloc[0]['article'])

# Remover plantillas
df_top_3 = remover_plantillas(df_top_3)

# Imprimir un ejemplo de articulo no antes de remover plantillas y luego de remover plantillas de Reuters
print("Ejemplo de artículo después de remover plantillas:")
print(df_top_3.iloc[0]['article'])

df_top_3["CleanText"] = clean_text(df_top_3["article"])

# Imprimir el articulo luego de clean_text
print("Ejemplo de artículo después de clean_text:")
print(df_top_3.iloc[0]['CleanText'])


In [ ]:
# TODO: Particione los datos en train y test (30% test), con muestreo estratificado
X = df_top_3["CleanText"]
y = df_top_3["publication"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

## 2. Verificación del balance de clases
Genere una visualización que permita verificar que el balance de artículos de cada medio es similar en train y test.

In [ ]:
# Combinar y_train y y_test para facilitar la creación de gráficos

# Convertirlos primero a DataFrames yagregar una columna 'Set'
y_train_df = y_train.to_frame(name='publication')
y_train_df['Set'] = 'Train'

y_test_df = y_test.to_frame(name='publication')
y_test_df['Set'] = 'Test'

combined_y_df = pd.concat([y_train_df, y_test_df])

# Calcular la cantidad de artículos por publicación y conjunto (train/test)
counts_by_publication_and_set = combined_y_df.groupby(['publication', 'Set']).size().unstack(fill_value=0)
print("Cantidad de artículos por publicación en los conjuntos de entrenamiento y prueba:\n")
print(counts_by_publication_and_set)

# Crear un gráfico de barras para visualizar la distribución de artículos por publicación y conjunto
plt.figure(figsize=(10, 6))
sns.countplot(data=combined_y_df, x='publication', hue='Set', palette='viridis')
plt.title('Distribución de Publicaciones en Conjuntos de Entrenamiento y Prueba')
plt.xlabel('Publicación')
plt.ylabel('Cantidad de Artículos')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Conjunto')
plt.tight_layout()
plt.show()

## 3. Representación Bag of Words
Transforme el texto del conjunto de entrenamiento a una representación numérica (features) de conteo de palabras (*bag of words*).
Explique brevemente cómo funciona esta técnica y muestre un ejemplo.
En particular, explique el tamaño de la matriz resultante y la razón por la que es una matriz *sparse*.

**Sugerencia**: puede ser útil imaginar qué sucedería con la memoria RAM requerida si no estuviéramos trabajando con un conjunto de datos reducido.

In [ ]:
# Inicializar CountVectorizer
vectorizer = CountVectorizer()

# Preprocesar X_train para manejar valores nulos
X_train_processed = X_train.fillna('')

# Transformar X_train_processed en una matriz de Bag of Words
X_train_bow = vectorizer.fit_transform(X_train_processed)
print(f"Tamaño de la matriz Bag of Words: {X_train_bow.shape}")

# Obterner las palabras del primer documento y sus conteos
first_document_index = 0
first_document_bow = X_train_bow[first_document_index].toarray()
vocabulary = vectorizer.get_feature_names_out()
document_word_counts = {}
for word_index in first_document_bow.nonzero()[1]:
    word = vocabulary[word_index]
    count = first_document_bow[0, word_index]
    document_word_counts[word] = count

print(f"\nEjemplo del documento {first_document_index}:")
print(f"Texto original: {X_train_processed.iloc[first_document_index][:1000]}...") # Show first 200 chars
print(f"Recuentos de palabras (primeras 20 palabras): {dict(list(document_word_counts.items())[:20])}")

## 4. Representación TF-IDF
Explique brevemente qué es un **n-grama**.
Obtenga la representación numérica *Term Frequency - Inverse Document Frequency* (TF-IDF).
Explique brevemente en qué consiste esta transformación adicional.

In [ ]:
X_train_processed = clean_text(X_train)

tfidf_vectorizer = TfidfVectorizer(
    use_idf=True,
    stop_words='english',
    ngram_range=(1,1)
)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_processed)

# Show the size of the resulting matrix
print(f"Shape of the TF-IDF matrix: {X_train_tfidf.shape}")

print(tfidf_vectorizer.get_feature_names_out()[:20])

In [ ]:
first_document_index = 0

# Obtener el texto original del primer documento
original_text_first_doc = X_train_processed.iloc[first_document_index]

# Obtener los nombres de las características (vocabulario) del vectorizador TF-IDF
vocabulary = tfidf_vectorizer.get_feature_names_out()

# Obtener los índices y puntajes de los valores TF-IDF no nulos para el primer documento
non_zero_indices = X_train_tfidf[first_document_index].nonzero()[1]
non_zero_scores = X_train_tfidf[first_document_index, non_zero_indices].toarray().flatten()

# Crear una lista de tuplas (palabra, puntaje) para el primer documento
word_score_pairs = []
for i, idx in enumerate(non_zero_indices):
    word = vocabulary[idx]
    score = non_zero_scores[i]
    word_score_pairs.append((word, score))

# Ordenar la lista de tuplas por puntaje TF-IDF en orden descendente
word_score_pairs.sort(key=lambda x: x[1], reverse=True)

print(f"\nMayores 20 valores TF-IDF para el documento {first_document_index}:")
# Mostrar las 20 palabras con mayor puntaje TF-IDF y sus puntajes
for i, (word, score) in enumerate(word_score_pairs):
    if i >= 20:
        break
    print(f"  {word}: {score:.4f}")

## 5. Visualización PCA sobre TF-IDF
Muestre en un mapa el conjunto de entrenamiento, utilizando las dos primeras componentes PCA sobre los vectores de TF-IDF.
Analice los resultados y compare qué sucede si utiliza:
- a) el filtrado de `stop_words` para idioma inglés;
- b) el parámetro `use_idf=True`;
- c) `ngram_range=(1,2)`.

Opcionalmente, también puede analizar qué sucede si no elimina los signos de puntuación.

¿Se pueden separar los medios de prensa utilizando sólo 2 componentes principales?
Haga una visualización que permita entender cómo varía la varianza explicada a medida que se agregan componentes (por ejemplo, hasta 10 componentes).

Discuta además si la separación observada puede deberse a diferencias de estilo editorial, a diferencias temáticas o a pistas explícitas del medio que no hayan sido removidas en la limpieza.

In [ ]:
# TODO: Compare los resultados de PCA con distintas configuraciones del TfidfVectorizer:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA

def plot_tfidf_pca(
    X,
    y,
    use_idf=False,
    stop_words=None,
    ngram_range=(1, 1),
    n_components=2,
    random_state=42,
    figsize=(10, 8)
):
    """
    Aplica TF-IDF y PCA sobre un conjunto de textos y grafica los resultados.
    """

    # Limpiar NaNs
    X_processed = X.fillna('')

    # TF-IDF
    vectorizer = TfidfVectorizer(
        use_idf=use_idf,
        stop_words=stop_words,
        ngram_range=ngram_range,
    )

    X_tfidf = vectorizer.fit_transform(X_processed)

    # PCA
    pca = PCA(
        n_components=n_components,
        random_state=random_state
    )

    X_pca = pca.fit_transform(X_tfidf.toarray())

    # DataFrame para plot
    pca_df = pd.DataFrame(
        X_pca,
        columns=[f'PC{i+1}' for i in range(n_components)]
    )

    pca_df['label'] = y.reset_index(drop=True)

    # Plot (solo si hay 2 componentes)
    if n_components == 2:
        plt.figure(figsize=figsize)

        sns.scatterplot(
            data=pca_df,
            x='PC1',
            y='PC2',
            hue='label',
            alpha=0.7
        )

        plt.title('PCA sobre TF-IDF')
        plt.grid(True)
        plt.tight_layout()
        plt.show()

    return {
        "vectorizer": vectorizer,
        "X_tfidf": X_tfidf,
        "pca": pca,
        "X_pca": X_pca,
        "explained_variance": pca.explained_variance_ratio_
    }

In [ ]:
result = plot_tfidf_pca(
    X_train,
    y_train,
    use_idf=False,
    stop_words=None,
    ngram_range=(1,1)
)

In [ ]:
result = plot_tfidf_pca(
    X_train,
    y_train,
    use_idf=True,
    stop_words=None,
    ngram_range=(1,1)
)

In [ ]:
result = plot_tfidf_pca(
    X_train,
    y_train,
    use_idf=False,
    stop_words='english',
    ngram_range=(1,1)
)

In [ ]:
result = plot_tfidf_pca(
    X_train,
    y_train,
    use_idf=False,
    stop_words=None,
    ngram_range=(1,2)
)

In [ ]:
# TODO: Genere una visualización que muestre cómo varía la varianza explicada
# a medida que se agregan componentes PCA (por ejemplo, hasta 10 componentes).

In [ ]:
pca_full = PCA(n_components=10, random_state=42)
X_train_pca_full = pca_full.fit_transform(X_train_tfidf.toarray())

# Calcular la varianza explicada acumulada
cum_var_exp = np.cumsum(pca_full.explained_variance_ratio_)

# Graficar la varianza explicada acumulada
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(cum_var_exp) + 1), cum_var_exp, marker='o', linestyle='--')
plt.title('Varianza Explicada Acumulada por Número de Componentes Principales')
plt.xlabel('Número de Componentes Principales')
plt.ylabel('Varianza Explicada Acumulada')
plt.grid(True)
plt.xticks(range(1, len(cum_var_exp) + 1))
plt.tight_layout()
plt.show()

# Parte 2: Entrenamiento y Evaluación de Modelos

## 1. Multinomial Naive Bayes
Entrene el modelo *Multinomial Naive Bayes* para clasificar los artículos según a qué medio de prensa pertenece el texto.
Utilice dicho modelo para clasificar los artículos del conjunto de test, y reporte el valor de *accuracy* y la **matriz de confusión**.
Reporte además el valor de *precision* y *recall* para cada medio.
Explique cómo se relacionan estos valores con la matriz anterior.

¿Qué problemas puede tener el hecho de mirar solamente el valor de *accuracy*?
Considere qué sucedería con esta métrica si el desbalance de datos fuera aún mayor entre medios.

**Sugerencia**: utilice el método `from_predictions` de `ConfusionMatrixDisplay` para realizar la matriz.

In [ ]:
# TODO: Entrene Multinomial Naive Bayes sobre X_train_tfidf

mnb = MultinomialNB()
mnb.fit(X_train_tfidf, y_train)

# Preprocesar X_test para manejar valores nulos
X_test_processed = X_test.fillna('')

# Transformar X_test usando el tfidf_vectorizer ajustado
X_test_tfidf = tfidf_vectorizer.transform(X_test_processed)

# Hacer predicciones en el conjunto de prueba
y_pred = mnb.predict(X_test_tfidf)

# Calcular accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

# Gráfica de la matriz de confusión
fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap=plt.cm.Blues, ax=ax)
ax.set_title('Confusion Matrix for Multinomial Naive Bayes')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Imprimir el reporte de clasificación
print("\nReporte de Clasificación:")
print(classification_report(y_test, y_pred))

## 2. Validación cruzada y búsqueda de hiperparámetros
Explique cómo funciona la técnica de **validación cruzada** (*cross-validation*).
Implemente una búsqueda de hiperparámetros usando `GridSearchCV`.
Genere una visualización que permita comparar las métricas (por ejemplo, *accuracy*) de los distintos modelos entrenados, viendo el valor promedio y la variabilidad de las mismas en todos los *splits* (por ejemplo, en un gráfico de violín).

In [ ]:
param_grid = {
    'alpha': [0.01, 0.1, 0.5, 1.0, 5.0, 10.0]  # Parametros de suavizado
}

# Inicializar el modelo Multinomial Naive Bayes
mnb = MultinomialNB()

# Inicializar GridSearchCV
grid_search = GridSearchCV(
    mnb,
    param_grid,
    cv=5,  # 5-fold cross-validation
    scoring='accuracy',  # Metric to optimize
    n_jobs=-1,  # Use all available CPU cores
    verbose=1
)

# Ajustar GridSearchCV a los datos de entrenamiento
grid_search.fit(X_train_tfidf, y_train)

print(f"Mejores parámetros encontrados: {grid_search.best_params_}")
print(f"Mejor accuracy en validación cruzada: {grid_search.best_score_:.4f}")

In [ ]:
# Extraer los resultados de GridSearchCV
results = grid_search.cv_results_

# Preparar los datos para la visualización
alpha_values = results['param_alpha']
mean_test_scores = results['mean_test_score']
std_test_scores = results['std_test_score']

# Crear un DataFrame para la visualización
plot_data = []
for i, alpha in enumerate(alpha_values):
    for split_idx in range(grid_search.cv):
        split_score_key = f'split{split_idx}_test_score'
        if split_score_key in results:
            plot_data.append({
                'alpha': alpha,
                'accuracy': results[split_score_key][i],
                'split': split_idx
            })

scores_df = pd.DataFrame(plot_data)

# Convertir 'alpha' a tipo string para evitar advertencias de FutureWarning en seaborn
scores_df['alpha'] = scores_df['alpha'].astype(str)

plt.figure(figsize=(10, 6))
sns.violinplot(x='alpha', y='accuracy', data=scores_df, hue='alpha', palette='viridis', legend=False) # Fixed FutureWarning

# Calcular la media de accuracy para cada valor de alpha
mean_accuracies = scores_df.groupby('alpha')['accuracy'].mean().reset_index()

# Convertir 'alpha' a tipo string para mean_accuracies también
mean_accuracies['alpha'] = mean_accuracies['alpha'].astype(str)

# Agregar los puntos de media sobre el gráfico de violín
sns.scatterplot(
    x='alpha',
    y='accuracy',
    data=mean_accuracies,
    color='red', 
    marker='X', 
    s=15, # Tamaño del marcador
    zorder=5, # Asegura que los puntos estén por encima del gráfico de violín
    label='Mean Accuracy'
)

plt.title('Distribución de Accuracy por Valor de Alpha (Smoothing Parameter)')
plt.xlabel('Alpha (Smoothing Parameter)')
plt.ylabel('Accuracy')
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend(title='Metric')
plt.tight_layout()
plt.show()

## 3. Entrenamiento final con el mejor modelo
Elija el mejor modelo (mejores parámetros) y vuelva a entrenar sobre todo el conjunto de entrenamiento disponible (sin quitar datos para validación).
Reporte el valor final de las métricas y la matriz de confusión.
Discuta las limitaciones de utilizar un modelo basado en *bag of words* o TF-IDF para el análisis de texto.

In [ ]:
# Obtener el mejor valor de alpha del GridSearchCV
best_alpha = grid_search.best_params_['alpha']

# Inicializar el modelo final de Multinomial Naive Bayes con el mejor alpha
final_mnb_model = MultinomialNB(alpha=best_alpha)

# Entreinar el modelo final con los datos de entrenamiento
final_mnb_model.fit(X_train_tfidf, y_train)

print(f"Modelo Multinomial Naive Bayes final entrenado con alpha={best_alpha}")

In [ ]:
# Predicciones en el conjunto de prueba usando el modelo final
y_pred_final = final_mnb_model.predict(X_test_tfidf)

# Calcular la precisión final en el conjunto de prueba
accuracy_final = accuracy_score(y_test, y_pred_final)
print(f"Precisión final del modelo en el conjunto de prueba: {accuracy_final:.4f}")

# Gráfica de la matriz de confusión para el modelo final
fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_final, cmap=plt.cm.Blues, ax=ax)
ax.set_title('Matriz de Confusión para el Modelo Multinomial Naive Bayes Final')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Mostrar el informe de clasificación para el modelo final
print("\nInforme de Clasificación para el Modelo Final:")
print(classification_report(y_test, y_pred_final))

## 4. Modelo alternativo
Evalúe al menos un modelo más (dentro de scikit-learn) aparte de *Multinomial Naive Bayes* para clasificar el texto utilizando las mismas *features* de texto.
Explique brevemente cómo funciona y compare los resultados con el anterior.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Inicializar el modelo de Regresión Logística
lr_model = LogisticRegression(
    solver='lbfgs',
    max_iter=1000,
    random_state=42
)

# Entrenar el modelo con los datos de entrenamiento
lr_model.fit(X_train_tfidf, y_train)

# Realizar predicciones sobre el conjunto de prueba
y_pred_lr = lr_model.predict(X_test_tfidf)

# Calcular la exactitud (accuracy)
accuracy_lr = accuracy_score(y_test, y_pred_lr)

print(
    f"Exactitud del modelo de Regresión Logística "
    f"en el conjunto de prueba: {accuracy_lr:.4f}"
)

# Mostrar la matriz de confusión
fig, ax = plt.subplots(figsize=(10, 8))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_lr,
    cmap=plt.cm.Greens,
    ax=ax
)

ax.set_title('Matriz de Confusión para el Modelo de Regresión Logística')

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Mostrar el informe de clasificación
print("\nInforme de clasificación para el modelo de Regresión Logística:")
print(classification_report(y_test, y_pred_lr))

## 5. Cambio de medio de prensa
Evalúe el problema cambiando al menos un medio de prensa.
En particular, observe el (des)balance de datos y los problemas que pueda generar, así como cualquier indicio que pueda ver en el mapeo previo con PCA.
Puede ser útil comentar acerca de técnicas como sobre-muestreo y submuestreo; no es necesario implementarlas.

## Re-evaluación con Reuters, The New York Times y The Hill

In [ ]:
# Re-evaluación del modelo con Reuters, The New York Times y The Hill.

print("Re-evaluación del modelo con Reuters, The New York Times y The Hill")   

# Nueva selección de publicaciones para el análisis
new_publications_hill_1 = ['Reuters', 'The New York Times', 'The Hill']

# Filtrar el DataFrame original para incluir solo las publicaciones seleccionadas
df_new_selection_hill_1 = df[df['publication'].isin(new_publications_hill_1)].copy()

# Verificar si 'The Hill' está presente en el DataFrame filtrado
if not 'The Hill' in df_new_selection_hill_1['publication'].unique():
    print("'The Hill' no está presente en el conjunto de datos filtrado")
    print(f"Publicaciones disponibles en el conjunto de datos filtrado: {df_new_selection_hill_1['publication'].unique()}")
else:
    print(f"Publicaciones seleccionadas: {df_new_selection_hill_1['publication'].unique()}")

# Aplicar clean_text sobre la columna de texto elegida y crear una nueva columna "CleanText"
df_new_selection_hill_1["CleanText"] = clean_text(df_new_selection_hill_1["article"])

# Eliminar las filas con valores nulos en la columna "CleanText" para evitar problemas durante el entrenamiento del modelo
df_new_selection_hill_1.dropna(subset=["CleanText"], inplace=True)

print(f"\nNumero de artículos en la nueva selección: {len(df_new_selection_hill_1)}")

# Análisis del balance de clases en este nuevo conjunto de datos 
X_new_hill_1 = df_new_selection_hill_1["CleanText"]
y_new_hill_1 = df_new_selection_hill_1["publication"]

# Imprimir el balance de clases antes de la partición
print("\nBalance de clases en el nuevo conjunto de datos (antes de split):")
print(y_new_hill_1.value_counts())

# Particionar los datos en train y test (30% test), con muestreo estratificado
X_train_new_hill_1, X_test_new_hill_1, y_train_new_hill_1, y_test_new_hill_1 = train_test_split(
    X_new_hill_1, y_new_hill_1, test_size=0.3, stratify=y_new_hill_1, random_state=42
)

# Combinar y_train_new_hill_1 y y_test_new_hill_1 para facilitar la creación de gráficos
y_train_df_new_hill_1 = y_train_new_hill_1.to_frame(name='publication')
y_train_df_new_hill_1['Set'] = 'Train'

y_test_df_new_hill_1 = y_test_new_hill_1.to_frame(name='publication')
y_test_df_new_hill_1['Set'] = 'Test'

combined_y_df_new_hill_1 = pd.concat([y_train_df_new_hill_1, y_test_df_new_hill_1])

# Calcular la cantidad de artículos por publicación y conjunto (train/test) para el nuevo conjunto
counts_by_publication_and_set_new_hill_1 = combined_y_df_new_hill_1.groupby(['publication', 'Set']).size().unstack(fill_value=0)
print("\nCantidad de artículos por publicación en los nuevos conjuntos de entrenamiento y prueba:\n")
print(counts_by_publication_and_set_new_hill_1)

# Crear un gráfico de barras para visualizar la distribución de artículos por publicación y conjunto en el nuevo conjunto
plt.figure(figsize=(10, 6))
sns.countplot(data=combined_y_df_new_hill_1, x='publication', hue='Set', palette='viridis')
plt.title('Distribución de Nuevas Publicaciones en Conjuntos de Entrenamiento y Prueba (The Hill)')
plt.xlabel('Publicación')
plt.ylabel('Cantidad de Artículos')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Conjunto')
plt.tight_layout()
plt.show()

# Vectorización TF-IDF para el nuevo conjunto de datos (The Hill)
tfidf_vectorizer_new_hill_1 = TfidfVectorizer(use_idf=True, stop_words='english')

# Preprocesar X_train_new_hill_1 y X_test_new_hill_1 para manejar valores nulos
X_train_processed_new_hill_1 = X_train_new_hill_1.fillna('')
X_test_processed_new_hill_1 = X_test_new_hill_1.fillna('')

# Transformar los conjuntos de entrenamiento y prueba usando el vectorizador TF-IDF ajustado
X_train_tfidf_new_hill_1 = tfidf_vectorizer_new_hill_1.fit_transform(X_train_processed_new_hill_1)
X_test_tfidf_new_hill_1 = tfidf_vectorizer_new_hill_1.transform(X_test_processed_new_hill_1)

print(f"\nTamaño de la nueva matriz TF-IDF (Entrenamiento): {X_train_tfidf_new_hill_1.shape}")
print(f"Tamaño de la nueva matriz TF-IDF (Prueba): {X_test_tfidf_new_hill_1.shape}")

# Entrenamiento y evaluación del modelo Multinomial Naive Bayes con el nuevo conjunto de datos (The Hill)
# Initializar el modelo Multinomial Naive Bayes con un valor de alpha ajustado
mnb_new_hill_1 = MultinomialNB(alpha=0.1)

# Entrenar el modelo con los datos de entrenamiento del nuevo conjunto
mnb_new_hill_1.fit(X_train_tfidf_new_hill_1, y_train_new_hill_1)

# Hacer predicciones en el conjunto de prueba del nuevo conjunto
y_pred_new_hill_1 = mnb_new_hill_1.predict(X_test_tfidf_new_hill_1)

# Calcular la precisión del modelo con el nuevo conjunto de datos (The Hill)
accuracy_new_hill_1 = accuracy_score(y_test_new_hill_1, y_pred_new_hill_1)
print(f"\nPrecisión del Multinomial Naive Bayes con nuevas publicaciones (The Hill) en el conjunto de prueba: {accuracy_new_hill_1:.4f}")

# Matriz de confusión para el modelo Multinomial Naive Bayes con el nuevo conjunto de datos (The Hill)
fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay.from_predictions(y_test_new_hill_1, y_pred_new_hill_1, cmap=plt.cm.Blues, ax=ax)
ax.set_title('Matriz de Confusión para Multinomial Naive Bayes (The Hill)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Reporte de clasificación para el modelo Multinomial Naive Bayes con el nuevo conjunto de datos (The Hill)
print("\nReporte de clasificación para el Multinomial Naive Bayes (The Hill):")
print(classification_report(y_test_new_hill_1, y_pred_new_hill_1))


## Re-evaluación con The Hill, The New York Times y CNBC

In [ ]:
# Re-evaluación del modelo con The Hill, The New York Times y CNBC

print("Re-evaluación del modelo con The Hill, The New York Times y CNBC")

# Nuevo conjunto de publicaciones: The Hill, The New York Times y CNBC
new_publications_hill_cnbc = ['The Hill', 'The New York Times', 'CNBC']

# Filtrar el DataFrame original para estas publicaciones
df_new_selection_hill_cnbc = df[df['publication'].isin(new_publications_hill_cnbc)].copy()

# Verificar si The Hill está presente en el conjunto de datos
if not 'The Hill' in df_new_selection_hill_cnbc['publication'].unique():
    print("Advertencia: 'The Hill' no fue encontrado en el conjunto de datos. Verifique los nombres de las publicaciones.")
    print(f"Publicaciones disponibles en el conjunto filtrado: {df_new_selection_hill_cnbc['publication'].unique()}")
else:
    print(f"Publicaciones seleccionadas: {df_new_selection_hill_cnbc['publication'].unique()}")

# Aplicar clean_text sobre la columna de texto y crear la columna CleanText
df_new_selection_hill_cnbc["CleanText"] = clean_text(df_new_selection_hill_cnbc["article"])

# Eliminar filas donde CleanText sea nulo o esté vacío luego de la limpieza
df_new_selection_hill_cnbc.dropna(subset=["CleanText"], inplace=True)

print(f"\nCantidad de artículos en la nueva selección: {len(df_new_selection_hill_cnbc)}")

# --- Análisis del balance de clases en el nuevo conjunto de datos ---
X_new_hill_cnbc = df_new_selection_hill_cnbc["CleanText"]
y_new_hill_cnbc = df_new_selection_hill_cnbc["publication"]

# Mostrar balance de clases antes de la partición
print("\nBalance de clases en el nuevo conjunto de datos (antes de la partición):")
print(y_new_hill_cnbc.value_counts())

# Dividir los datos en entrenamiento y prueba (30% prueba) usando muestreo estratificado
X_train_new_hill_cnbc, X_test_new_hill_cnbc, y_train_new_hill_cnbc, y_test_new_hill_cnbc = train_test_split(
    X_new_hill_cnbc, y_new_hill_cnbc, test_size=0.3, stratify=y_new_hill_cnbc, random_state=42
)

# Combinar entrenamiento y prueba para facilitar la visualización
y_train_df_new_hill_cnbc = y_train_new_hill_cnbc.to_frame(name='publication')
y_train_df_new_hill_cnbc['Set'] = 'Entrenamiento'

y_test_df_new_hill_cnbc = y_test_new_hill_cnbc.to_frame(name='publication')
y_test_df_new_hill_cnbc['Set'] = 'Prueba'

combined_y_df_new_hill_cnbc = pd.concat([y_train_df_new_hill_cnbc, y_test_df_new_hill_cnbc])

# Calcular y mostrar la cantidad de artículos por publicación y conjunto
counts_by_publication_and_set_new_hill_cnbc = combined_y_df_new_hill_cnbc.groupby(
    ['publication', 'Set']
).size().unstack(fill_value=0)

print("\nCantidad de artículos por publicación en los conjuntos de entrenamiento y prueba:\n")
print(counts_by_publication_and_set_new_hill_cnbc)

# Gráfico para verificar visualmente la distribución
plt.figure(figsize=(10, 6))
sns.countplot(
    data=combined_y_df_new_hill_cnbc,
    x='publication',
    hue='Set',
    palette='viridis'
)
plt.title('Distribución de publicaciones en los conjuntos de entrenamiento y prueba')
plt.xlabel('Publicación')
plt.ylabel('Cantidad de artículos')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Conjunto')
plt.tight_layout()
plt.show()

# --- Vectorización TF-IDF para el nuevo conjunto de datos ---

# Inicializar nuevamente el vectorizador TF-IDF
tfidf_vectorizer_new_hill_cnbc = TfidfVectorizer(
    use_idf=True,
    stop_words='english'
)

# Reemplazar posibles valores nulos por cadenas vacías
X_train_processed_new_hill_cnbc = X_train_new_hill_cnbc.fillna('')
X_test_processed_new_hill_cnbc = X_test_new_hill_cnbc.fillna('')

# Ajustar el vectorizador con entrenamiento y transformar ambos conjuntos
X_train_tfidf_new_hill_cnbc = tfidf_vectorizer_new_hill_cnbc.fit_transform(
    X_train_processed_new_hill_cnbc
)

X_test_tfidf_new_hill_cnbc = tfidf_vectorizer_new_hill_cnbc.transform(
    X_test_processed_new_hill_cnbc
)

print(f"\nDimensión de la matriz TF-IDF (entrenamiento): {X_train_tfidf_new_hill_cnbc.shape}")
print(f"Dimensión de la matriz TF-IDF (prueba): {X_test_tfidf_new_hill_cnbc.shape}")

# --- Entrenamiento y evaluación del modelo Multinomial Naive Bayes ---

# Inicializar el modelo con el mejor valor de alpha encontrado previamente
mnb_new_hill_cnbc = MultinomialNB(alpha=0.1)

# Entrenar el modelo
mnb_new_hill_cnbc.fit(X_train_tfidf_new_hill_cnbc, y_train_new_hill_cnbc)

# Realizar predicciones sobre el conjunto de prueba
y_pred_new_hill_cnbc = mnb_new_hill_cnbc.predict(X_test_tfidf_new_hill_cnbc)

# Calcular exactitud (accuracy)
accuracy_new_hill_cnbc = accuracy_score(
    y_test_new_hill_cnbc,
    y_pred_new_hill_cnbc
)

print(
    f"\nExactitud de Multinomial Naive Bayes con las nuevas publicaciones "
    f"(The Hill, The New York Times y CNBC): {accuracy_new_hill_cnbc:.4f}"
)

# Mostrar matriz de confusión
fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay.from_predictions(
    y_test_new_hill_cnbc,
    y_pred_new_hill_cnbc,
    cmap=plt.cm.Blues,
    ax=ax
)

ax.set_title('Matriz de confusión para Multinomial Naive Bayes')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Mostrar reporte de clasificación
print("\nReporte de clasificación para Multinomial Naive Bayes:")
print(classification_report(y_test_new_hill_cnbc, y_pred_new_hill_cnbc))

## 6. Técnica alternativa de extracción de features
Busque información sobre al menos una técnica alternativa de extraer *features* de texto.
Explique brevemente cómo funciona y qué tipo de diferencias esperaría en los resultados.
No se espera que implemente nada en esta parte.

*TODO: Escriba su análisis en el informe.*

In [ ]:
# Opcional: Repita el pipeline de clasificación utilizando el título del artículo
# en lugar del cuerpo del texto.